# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² mental health survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, as referenced below.

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print basic metadata
ds_metadata = dataset.metadata
print(f"Dataset name: {ds_metadata.name}\nDescription: {ds_metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and fields, referencing their @id attributes
print('Available Record Sets:')
for record_set in dataset.metadata.record_sets:
    print(f"- Name: {record_set.name}\n  @id: {record_set.id}\n  Description: {getattr(record_set, 'description', '')}")
    print('  Fields:')
    for field in record_set.fields:
        print(f"    - Field: {field.name}\n      @id: {field.id}\n      Data type: {field.data_type}\n      Description: {getattr(field, 'description', '')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, there is typically one main record set. List all record set @ids for extraction.
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f'Record set: {record_set_id}')
    print(df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, grouping, and outlier removal to demonstrate typical EDA workflows on the data.

In [ ]:
import numpy as np
main_record_set_id = record_set_ids[0]  # Use main record set
df = dataframes[main_record_set_id]

# For illustration, select the GAD-7 total score field if present
# You can list all numeric-like fields for further analysis
print('Attempting to find suitable numeric fields for EDA...')
numeric_field_candidates = [col for col in df.columns if df[col].dtype in [np.int64, np.float64] or np.issubdtype(df[col].dtype, np.number)]
if not numeric_field_candidates:
    # Try common total score names
    for col in df.columns:
        if any(score in col.lower() for score in ['gad', 'phq', 'psq', 'total']):
            numeric_field_candidates.append(col)
print(f"Numeric fields found: {numeric_field_candidates}")

if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]  # e.g., 'gad7_total'
    print(f'Using numeric field for analysis: {numeric_field}')

    # Remove rows with missing values for the selected numeric field
    df_num = df.dropna(subset=[numeric_field])
    # Convert to numeric type if needed
    df_num[numeric_field] = pd.to_numeric(df_num[numeric_field], errors='coerce')

    threshold = df_num[numeric_field].mean()  # For demonstration, use mean as threshold
    filtered_df = df_num[df_num[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field, e.g., 'gender' if present
    group_field = None
    for col in df.columns:
        if col.lower() in ['gender', 'sex', 'village', 'level_of_education', 'marital_status']:
            group_field = col
            break
    if group_field:
        print(f"Grouping data by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'std', 'count'])
        display(grouped_df.head())
    else:
        print('No obvious categorical field found for grouping.')
else:
    print('No numeric fields found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if numeric_field_candidates:
    # Histogram of selected numeric field
    plt.figure(figsize=(8,4))
    df[numeric_field].dropna().astype(float).hist(bins=20)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by category if group_field exists
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the FAIR² Mental Health Survey dataset from a Croissant schema using `mlcroissant`.
- Explored available record sets and fields referenced by their `@id`s.
- Loaded data into Pandas DataFrames for each record set.
- Performed basic EDA: filtered and normalized a selected numeric assessment field, and explored relationships by demographic group where available.
- Visualized key data distributions.

This approach can be adapted to other Croissant-based datasets for rapid, standards-based data science workflows.